# TA-07 — Simulasi Sistem Dinamik: Model Logistik Pertumbuhan Bakteri
**Mata Kuliah:** Pemodelan dan Simulasi | **Nama/NIM:** [Nama Lengkap] / [NIM]

## 1. Penjelasan Model
Stock: populasi bakteri $P(t)$. Inflow $rP$ (feedback positif), pembatas $rP^2/K$ (feedback negatif).

$$\frac{dP}{dt} = rP\left(1-\frac{P}{K}\right),\quad P(0)=P_0$$

- $r$ = laju pertumbuhan (per jam), $K$ = kapasitas dukung.
- Euler: $P_{t+h}=P_t+h\,f(t,P_t)$ | RK4 | `solve_ivp` (adaptif).

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
plt.rcParams['axes.grid']=True

## 2. Data dan EDA

In [ ]:
# Jika di Colab: from google.colab import files; files.upload()
df_raw = pd.read_csv("bacterial_growth.csv")
row = df_raw.iloc[0]
df = pd.DataFrame({"Time": row.index.astype(float), "Population": row.values})
df["Jam"] = df["Time"]/3600          # detik -> jam
print(df.shape); print(df.dtypes); print("Missing:", df.Population.isnull().sum())
display(df.Population.describe())
plt.figure(figsize=(9,4)); plt.plot(df.Jam, df.Population,'o',ms=3)
plt.xlabel("Waktu (jam)"); plt.ylabel("Populasi"); plt.title("Data Pertumbuhan Bakteri"); plt.show()

**Interpretasi:** 89 titik, tanpa missing value, kurva S (sigmoid) dengan puncak ±13 jam lalu sedikit menurun.

## 3. Kalibrasi Parameter (r, K)
Parameter tutorial (r=0,8 pada skala detik) membuat simulasi langsung melonjak ke K, jadi r harus dikalibrasi sesuai satuan waktu.

In [ ]:
P0 = df.Population.iloc[0]
logi = lambda t, r, K: K/(1+(K-P0)/P0*np.exp(-r*t))   # solusi analitik
(r, K), _ = curve_fit(logi, df.Jam, df.Population, p0=[0.3,0.62])
rmse = lambda a,b: np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2))
print(f"r = {r:.4f} /jam | K = {K:.4f} | RMSE = {rmse(logi(df.Jam,r,K), df.Population):.4f}")

## 4. Implementasi Simulasi

In [ ]:
def f(t, P, r, K): return r*P*(1-P/K)

def euler(h, T, r=r, K=K):
    n=int(round(T/h)); t=np.linspace(0,n*h,n+1); y=np.zeros(n+1); y[0]=P0
    for i in range(n): y[i+1]=y[i]+h*f(t[i],y[i],r,K)
    return t, y

def rk4(h, T, r=r, K=K):
    n=int(round(T/h)); t=np.linspace(0,n*h,n+1); y=np.zeros(n+1); y[0]=P0
    for i in range(n):
        k1=f(t[i],y[i],r,K);        k2=f(t[i],y[i]+h/2*k1,r,K)
        k3=f(t[i],y[i]+h/2*k2,r,K); k4=f(t[i],y[i]+h*k3,r,K)
        y[i+1]=y[i]+h/6*(k1+2*k2+2*k3+k4)
    return t, y

T = df.Jam.max(); tt = np.linspace(0,T,400)
sol = solve_ivp(f,(0,T),[P0],args=(r,K),t_eval=tt,rtol=1e-8,atol=1e-10)

# Simpan hasil ke DataFrame
hasil = pd.DataFrame({"Jam": sol.t, "P_solve_ivp": sol.y[0]})
hasil.head()

In [ ]:
plt.figure(figsize=(9,4.5))
plt.plot(df.Jam, df.Population,'o',ms=3,color='gray',label="Data")
plt.plot(*euler(5,T),label="Euler h=5 jam"); plt.plot(*euler(1,T),label="Euler h=1 jam")
plt.plot(*rk4(1,T),'--',label="RK4 h=1 jam"); plt.plot(sol.t,sol.y[0],'r:',lw=2,label="solve_ivp")
plt.xlabel("Waktu (jam)"); plt.ylabel("Populasi"); plt.title("Perbandingan Metode Numerik"); plt.legend(); plt.show()

In [ ]:
# Pengaruh step size: galat Euler terhadap solusi analitik di t=T
for h in [5,2,1,0.5,0.1]:
    te,ye = euler(h,T); print(f"h={h:>4} jam | galat = {abs(ye[-1]-logi(te[-1],r,K)):.5f}")
print("RMSE vs data -> Euler h=1:", round(rmse(np.interp(df.Jam,*euler(1,T)),df.Population),4),
      "| RK4 h=1:", round(rmse(np.interp(df.Jam,*rk4(1,T)),df.Population),4),
      "| solve_ivp:", round(rmse(np.interp(df.Jam,sol.t,sol.y[0]),df.Population),4))

**Interpretasi:** galat Euler mengecil ≈ sebanding dengan h (orde 1). RK4 dan `solve_ivp` hampir berhimpit dengan solusi analitik pada h=1 jam.

## 5. Analisis Pola
Kurva berbentuk S-curve: tumbuh cepat saat P kecil, melambat mendekati K (feedback negatif), lalu stabil. Perbedaan dengan data: ada fase *lag* di awal dan penurunan setelah puncak yang tidak ada dalam model logistik.

In [ ]:
plt.figure(figsize=(9,4.5)); plt.plot(df.Jam, df.Population,'o',ms=3,color='gray',label="Data")
plt.plot(sol.t, sol.y[0],'r',label=f"Logistik r={r:.3f}, K={K:.3f}")
plt.xlabel("Waktu (jam)"); plt.ylabel("Populasi"); plt.title("Data vs Simulasi Terkalibrasi"); plt.legend(); plt.show()

## 6. Eksperimen What-If (Baseline vs Skenario)

In [ ]:
skenario = {"Baseline":(r,K), "r turun 50%":(r*0.5,K), "K naik 20%":(r,K*1.2), "K turun 20%":(r,K*0.8)}
plt.figure(figsize=(9,4.5)); plt.plot(df.Jam, df.Population,'o',ms=3,color='gray',label="Data")
ringkas=[]
for nama,(rr,KK) in skenario.items():
    s = solve_ivp(f,(0,T),[P0],args=(rr,KK),t_eval=tt)
    plt.plot(s.t,s.y[0],label=nama)
    idx = np.argmax(s.y[0]>=0.9*KK) if s.y[0].max()>=0.9*KK else None
    ringkas.append([nama, round(s.y[0][-1],3), round(s.t[idx],1) if idx is not None else "belum tercapai"])
plt.xlabel("Waktu (jam)"); plt.ylabel("Populasi"); plt.title("Skenario What-If"); plt.legend(); plt.show()
pd.DataFrame(ringkas, columns=["Skenario","Populasi akhir","Waktu 90% K (jam)"])

**Interpretasi:** r turun 50% → bentuk S tetap tetapi jauh lebih lambat (90% K tidak tercapai). K naik/turun → mengubah tinggi plateau, fase awal hampir sama karena saat P kecil $(1-P/K)\approx1$.

## 7. Kesimpulan & Keterbatasan
Model logistik meniru tren sigmoid dengan r≈0,308/jam, K≈0,631 (RMSE≈0,040). Keterbatasan: tidak memuat fase lag dan fase kematian, parameter konstan, satu stock saja.